# Mobilità sistematica interna al Friuli-Venezia Giulia

Il notebook costruisce una matrice origine-destinazione dei flussi di pendolarismo interni al Friuli-Venezia Giulia a partire dai dati ISTAT 2021. I codici territoriali sono decodificati mediante l'anagrafica nazionale dei comuni.

Sono incluse esclusivamente le relazioni nelle quali il comune di residenza e il comune di lavoro appartengono entrambi alla regione. Gli archi intra-comunali sono conservati.

In [ ]:
from pathlib import Path

import pandas as pd

DIR_INPUT = Path("input")
DIR_OUTPUT = Path("output")
DIR_OUTPUT_INTRA = DIR_OUTPUT / "intra_regione"
DIR_OUTPUT_EXTRA = DIR_OUTPUT / "extra_regione"
DIR_OUTPUT_INTRA.mkdir(parents=True, exist_ok=True)
DIR_OUTPUT_EXTRA.mkdir(parents=True, exist_ok=True)

FILE_MATRICE_OD = DIR_INPUT / "flussi_pendolarismo" / "matrix_pendoLAVORO_2021.txt"
FILE_COMUNI_FVG = (
    DIR_INPUT / "anagrafiche_comuni"
    / "FVG_Elenco_dei_codici_e_delle_denominazioni_delle_unit_territoriali_FVG.csv"
)
FILE_COMUNI_ITALIA = (
    DIR_INPUT / "anagrafiche_comuni"
    / "ITA_Elenco_dei_codici_e_delle_denominazioni_delle_unit_territoriali_ITA.csv"
)
FILE_OUTPUT = DIR_OUTPUT_INTRA / "flussi_mobilita_interni_FVG_2021.csv"
FILE_OUTPUT_EXTRA_REGIONE = (
    DIR_OUTPUT_EXTRA / "flussi_mobilita_extra_regione_FVG_2021.csv"
)

# I codici territoriali sono letti come stringhe per preservare gli zeri iniziali.
tipi_colonne_od = {
    "Prov_res": "string",
    "Procom_res": "string",
    "Prov_lav": "string",
    "Procom_lav": "string",
    "Pendolari": "Int64",
}

matrice_od = pd.read_csv(FILE_MATRICE_OD, sep="\t", dtype=tipi_colonne_od)
comuni_fvg = pd.read_csv(FILE_COMUNI_FVG, sep=";", dtype="string")
comuni_italia = pd.read_csv(FILE_COMUNI_ITALIA, sep=";", dtype="string")

## Selezione territoriale e controllo del join

L'elenco regionale identifica i comuni appartenenti al Friuli-Venezia Giulia. L'anagrafica nazionale associa a ogni codice comunale la relativa denominazione. Prima dell'unione sono verificati l'univocità dei codici e il numero di relazioni che non troverebbero corrispondenza.

In [ ]:
COLONNA_CODICE_COMUNE = "Codice Comune (alfanumerico)"

# L'anagrafica deve contenere una sola denominazione per ciascun codice comunale.
if comuni_italia[COLONNA_CODICE_COMUNE].duplicated().any():
    raise ValueError("L'anagrafica nazionale contiene codici comunali duplicati.")

codici_fvg = set(comuni_fvg[COLONNA_CODICE_COMUNE].dropna())
denominazioni_comuni = comuni_italia.set_index(COLONNA_CODICE_COMUNE)["Comune"]
codici_italia = set(denominazioni_comuni.index.dropna())

# Il controllo quantifica le osservazioni che un inner join eliminerebbe.
origine_non_decodificata = ~matrice_od["Procom_res"].isin(codici_italia)
destinazione_non_decodificata = ~matrice_od["Procom_lav"].isin(codici_italia)
relazione_non_decodificata = origine_non_decodificata | destinazione_non_decodificata

verifica_join = pd.DataFrame({
    "Indicatore": [
        "Origine non decodificata",
        "Destinazione non decodificata",
        "Relazione esclusa dall'inner join",
    ],
    "Numero_relazioni": [
        int(origine_non_decodificata.sum()),
        int(destinazione_non_decodificata.sum()),
        int(relazione_non_decodificata.sum()),
    ],
    "Pendolari": [
        matrice_od.loc[origine_non_decodificata, "Pendolari"].sum(),
        matrice_od.loc[destinazione_non_decodificata, "Pendolari"].sum(),
        matrice_od.loc[relazione_non_decodificata, "Pendolari"].sum(),
    ],
})
display(verifica_join)

# Sono selezionate le sole relazioni con origine e destinazione in FVG.
origine_fvg = matrice_od["Procom_res"].isin(codici_fvg)
destinazione_fvg = matrice_od["Procom_lav"].isin(codici_fvg)
flussi_fvg = matrice_od.loc[origine_fvg & destinazione_fvg].copy()

# Le denominazioni sono associate senza modificare i codici territoriali originali.
flussi_fvg["Comune_res"] = flussi_fvg["Procom_res"].map(denominazioni_comuni)
flussi_fvg["Comune_lav"] = flussi_fvg["Procom_lav"].map(denominazioni_comuni)

# I controlli garantiscono completezza, positività dei pesi e unicità degli archi.
if not flussi_fvg[["Comune_res", "Comune_lav", "Pendolari"]].notna().all().all():
    raise ValueError("I flussi interni contengono valori mancanti.")
if not flussi_fvg["Pendolari"].gt(0).all():
    raise ValueError("I flussi interni contengono pesi nulli o negativi.")
if flussi_fvg.duplicated(["Procom_res", "Procom_lav"]).any():
    raise ValueError("I flussi interni contengono chiavi OD duplicate.")

## Analisi descrittiva

La somma dei pesi misura i pendolari associati ai flussi interni alla regione. La distinzione tra archi intra-comunali e inter-comunali consente di separare chi lavora nel comune di residenza da chi si sposta verso un altro comune. I corridoi sono direzionali: una coppia origine-destinazione e il suo percorso inverso costituiscono due archi distinti.

In [ ]:
totale_pendolari = flussi_fvg["Pendolari"].sum()
statistiche_flussi = flussi_fvg["Pendolari"].describe().to_frame("Pendolari")

# Un arco è intra-comunale quando i codici di origine e destinazione coincidono.
arco_intra_comunale = flussi_fvg["Procom_res"].eq(flussi_fvg["Procom_lav"])
flussi_intra = flussi_fvg.loc[arco_intra_comunale].copy()
flussi_inter = flussi_fvg.loc[~arco_intra_comunale].copy()

riepilogo_flussi = pd.DataFrame({
    "Tipo_flusso": ["Intra-comunale", "Inter-comunale"],
    "Numero_archi_OD": [len(flussi_intra), len(flussi_inter)],
    "Pendolari": [
        flussi_intra["Pendolari"].sum(),
        flussi_inter["Pendolari"].sum(),
    ],
})
riepilogo_flussi["Percentuale_pendolari"] = (
    riepilogo_flussi["Pendolari"] / totale_pendolari * 100
).round(2)

# La graduatoria considera esclusivamente gli archi tra comuni differenti.
top_15_corridoi = (
    flussi_inter[["Procom_res", "Comune_res", "Procom_lav", "Comune_lav", "Pendolari"]]
    .sort_values("Pendolari", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
top_15_corridoi.index = top_15_corridoi.index + 1
top_15_corridoi.index.name = "Posizione"

print(f"Pendolari nei flussi interni al FVG: {totale_pendolari:,}")
display(statistiche_flussi)
display(riepilogo_flussi)
display(top_15_corridoi)

## Esportazione

Il dataset è ordinato per origine e destinazione ed esportato in formato CSV. Ogni riga rappresenta un arco diretto; `Pendolari` ne costituisce il peso.

In [ ]:
colonne_output = [
    "Prov_res",
    "Procom_res",
    "Comune_res",
    "Prov_lav",
    "Procom_lav",
    "Comune_lav",
    "Pendolari",
]

dataset_flussi_fvg = (
    flussi_fvg[colonne_output]
    .sort_values(["Procom_res", "Procom_lav"])
    .reset_index(drop=True)
)
dataset_flussi_fvg.to_csv(FILE_OUTPUT, index=False, encoding="utf-8-sig")

riepilogo_esportazione = pd.Series({
    "File": FILE_OUTPUT,
    "Archi OD": len(dataset_flussi_fvg),
    "Pendolari": dataset_flussi_fvg["Pendolari"].sum(),
}, name="Esportazione")
display(riepilogo_esportazione.to_frame())

## Flussi extra-regionali

Sono considerate extra-regionali le relazioni che hanno una sola estremità in Friuli-Venezia Giulia. I flussi in uscita hanno origine in FVG e destinazione nel resto d'Italia; i flussi in entrata seguono la direzione opposta. Le relazioni interne alla regione sono escluse da questo dataset.

In [ ]:
# L'operatore XOR seleziona le relazioni con una sola estremità in FVG.
flusso_extra_regionale = origine_fvg ^ destinazione_fvg
flussi_extra_regione = matrice_od.loc[flusso_extra_regionale].copy()

# Le denominazioni sono ricavate dall'anagrafica nazionale dei comuni.
flussi_extra_regione["Comune_res"] = (
    flussi_extra_regione["Procom_res"].map(denominazioni_comuni)
)
flussi_extra_regione["Comune_lav"] = (
    flussi_extra_regione["Procom_lav"].map(denominazioni_comuni)
)

# La direzione è definita rispetto al confine regionale.
origine_extra_fvg = flussi_extra_regione["Procom_res"].isin(codici_fvg)
flussi_extra_regione["Direzione"] = origine_extra_fvg.map(
    {True: "Uscita", False: "Entrata"}
)

# I controlli verificano decodifica, positività dei pesi e unicità degli archi.
if not flussi_extra_regione[["Comune_res", "Comune_lav", "Pendolari"]].notna().all().all():
    raise ValueError("I flussi extra-regionali contengono valori mancanti.")
if not flussi_extra_regione["Pendolari"].gt(0).all():
    raise ValueError("I flussi extra-regionali contengono pesi nulli o negativi.")
if flussi_extra_regione.duplicated(["Procom_res", "Procom_lav"]).any():
    raise ValueError("I flussi extra-regionali contengono chiavi OD duplicate.")

colonne_output_extra_regione = colonne_output + ["Direzione"]
dataset_flussi_extra_regione = (
    flussi_extra_regione[colonne_output_extra_regione]
    .sort_values(["Direzione", "Procom_res", "Procom_lav"])
    .reset_index(drop=True)
)
dataset_flussi_extra_regione.to_csv(
    FILE_OUTPUT_EXTRA_REGIONE, index=False, encoding="utf-8-sig"
)

# Il riepilogo separa consistenza e peso dei flussi in entrata e in uscita.
riepilogo_extra_regione = (
    dataset_flussi_extra_regione.groupby("Direzione", as_index=False)
    .agg(Numero_archi_OD=("Pendolari", "size"), Pendolari=("Pendolari", "sum"))
)
display(riepilogo_extra_regione)
display(pd.Series({
    "File": FILE_OUTPUT_EXTRA_REGIONE,
    "Archi OD": len(dataset_flussi_extra_regione),
    "Pendolari": dataset_flussi_extra_regione["Pendolari"].sum(),
}, name="Esportazione extra-regionale").to_frame())